In [ ]:
import re
import numpy as np
import pandas as pd

# ============================== ĐỌC DỮ LIỆU ==============================
FILE = "260718_So_lieu_lieu_phap_CART_VinUni_4chuAn.xlsx"
OUT = "cart_cleaned.csv"
SHEET_INFO, SHEET_CLIN, SHEET_FOLLOW = "Patient info", "Clinical Raw data", "CART and B cell follow up"

# PB = máu ngoại vi, BM = tủy xương
FOLLOW_FEATURES = {
    "B_cell_PB_cells_ml": "B cell depletion in PB (cells/ml)",
    "B_cell_PB_pct":      "Frequency of B cells in PB (%)",
    "CAR_T_PB_cells_ul":  "CAR T cell expansion in PB (cells/µl)",
    "CAR_T_PB_pct":       "Frequency of CAR T cells in PB (%)",
    "B_cell_BM_cells_ml": "B cell depletion in BM (cells/ml)",
    "B_cell_BM_pct":      "Frequency of B cells in BM (%)",
    "CAR_T_BM_cells_ul":  "CAR T cell expansion in BM (cells/µl)",
    "CAR_T_BM_pct":       "Frequency of CAR T cells in BM (%)",
}

# MRD và Blast tính ra %
CLIN_FEATURES = {
    "MRD_pct":   ({"MRD trước điều trị": 0, "MRD 1 tháng": 30, "MRD 3 tháng": 90, "MRD 6 tháng": 180}, 100),
    "Blast_pct": ({"Blast tủy trước điều trị": 0, "Blast tủy 1 tháng": 30, "Blast tủy 3 tháng": 90, "Blast tủy 6 tháng": 180}, 100),
}
IL6 = "IL6_pg_ml"
ALL_ONLY = ["MRD_pct", "Blast_pct"]          # chỉ có nghĩa với ALL, NHL là "không áp dụng"

FEATURES = list(FOLLOW_FEATURES) + list(CLIN_FEATURES) + [IL6]
COLS = ["Patient", "Day", "Diagnosis"] + FEATURES

INCLUDE_CRS_IL6 = True   # True: lấy thêm IL-6 lúc bắt đầu/kết thúc CRS; False: chỉ lấy baseline
ADD_MASKS = True         # thêm cột <biến>_obs (1 = có đo, 0 = thiếu) cho model biết chỗ nào là thật
ADD_DT = True            # thêm cột dt = số ngày kể từ lần ghi nhận trước (mốc đầu = 0), cần cho chuỗi không đều

# ============================== HỖ TRỢ ==============================
def parse_value(s: pd.Series) -> pd.Series:
    """Chuyển cột về float.
    - Ô đã là số thì giữ nguyên, không qua regex (tránh str(1e-05) bị đọc thành 1).
    - Ô là chuỗi: bỏ toán tử, đổi dấu phẩy thập phân, regex lấy số đầu tiên.
    - "<x" là dưới ngưỡng phát hiện nên lấy x/sqrt(2) (cách thay LOD hay dùng), vd "<1.5" ra ~1.06.
    - ">x" giữ nguyên x. Chuỗi không có số (vd "Bệnh tái phát") thì NaN."""
    direct = pd.to_numeric(s, errors="coerce")
    txt = s.where(direct.isna()).astype("string").str.strip().str.replace(",", ".", regex=False)
    num = pd.to_numeric(txt.str.extract(r"(-?\d+(?:\.\d+)?)", expand=False), errors="coerce")
    below_lod = txt.str.match(r"^(?:<|≤)", na=False)
    return direct.fillna(num.where(~below_lod, num / np.sqrt(2))).astype(float)

def norm(x) -> str:
    """Chuẩn hoá header để so khớp: bỏ hoa/thường, khoảng trắng, gộp µ/μ."""
    return re.sub(r"\s+", "", str(x).lower().replace("µ", "u").replace("μ", "u"))

def label_to_day(label):
    """'Before CAR T infusion' -> 0, 'Day 7+/-1' -> 7, 'Day 3/4' -> 3.
    'Prior apheresis' và 'Last visit' không có ngày cụ thể nên bỏ (None)."""
    s = str(label).lower()
    if "before" in s:
        return 0
    m = re.search(r"day\s*(\d+)", s)
    return int(m.group(1)) if m else None

def to_long(wide: pd.DataFrame, name: str) -> pd.DataFrame:
    """Bảng wide (PID + mỗi Day một cột) -> long [PID, Day, name] bằng melt."""
    out = wide.melt(id_vars="PID", var_name="Day", value_name=name)
    out[name] = parse_value(out[name])
    return out

# ============================== CHUẨN HOÁ ==============================
# Lấy Patient info làm gốc: PID -> BNxx + Diagnosis.
info = pd.read_excel(FILE, sheet_name=SHEET_INFO, header=3)[["PID", "Project ID", "Chẩn đoán"]]
info["PID"] = pd.to_numeric(info["PID"], errors="coerce").astype("Int64")
master = (info[info["Project ID"].astype(str).str.fullmatch(r"BN\d+")]
          .dropna(subset=["PID"]).drop_duplicates("PID")
          .rename(columns={"Project ID": "Patient", "Chẩn đoán": "Diagnosis"}))
pid2pat, pid2dx = master.set_index("PID")["Patient"], master.set_index("PID")["Diagnosis"]

# ============================== CLINICAL: IL-6 + MRD + Blast ==============================
clin = pd.read_excel(FILE, sheet_name=SHEET_CLIN)
clin["PID"] = pd.to_numeric(clin["PID"], errors="coerce").astype("Int64")
clin = clin[clin["PID"].isin(master["PID"])].reset_index(drop=True)

def clinical_long(col_to_day: dict, name: str, scale: float = 1.0) -> pd.DataFrame:
    w = clin[list(col_to_day)].apply(parse_value) * scale
    w.columns = list(col_to_day.values())
    w.insert(0, "PID", clin["PID"])
    return to_long(w, name)

clin_longs = [clinical_long(cols, name, scale) for name, (cols, scale) in CLIN_FEATURES.items()]

il6_parts = [clinical_long({"IL6 trước truyền CAR-T": 0}, IL6)]          # IL-6 baseline = Day 0
if INCLUDE_CRS_IL6:
    onset = pd.to_numeric(clin["Ngày phát hiện CRS"].astype("string").str.extract(r"(\d+)", expand=False))  # "D6" -> 6
    dur = pd.to_numeric(clin["Thời gian diễn ra CRS"].astype("string").str.extract(r"(\d+)", expand=False)) # "3 ngày" -> 3
    for col, day in [("IL6 khi bắt đầu CRS", onset), ("IL6 khi kết thúc CRS", onset + dur)]:
        part = pd.DataFrame({"PID": clin["PID"], "Day": day, IL6: parse_value(clin[col])})
        il6_parts.append(part.dropna(subset=["Day"]))                    # không bị CRS thì không có ngày -> bỏ qua
il6_long = pd.concat(il6_parts, ignore_index=True)

# ============================== FOLLOW-UP: B-cell + CAR-T (PB và BM) ==============================
raw = pd.read_excel(FILE, sheet_name=SHEET_FOLLOW, header=None)
hit = raw.iloc[:20].astype(str).apply(lambda c: c.str.strip().str.upper().eq("PID")).to_numpy()
r_pid, c_pid = np.argwhere(hit)[0]                       # tự dò dòng header chứa "PID" thay vì gõ cứng số dòng
metric_row = raw.iloc[r_pid - 1].ffill().map(norm)       # tên metric nằm ở ô merge nên phải ffill sang các cột bên cạnh
day_row = raw.iloc[r_pid].map(label_to_day)              # đổi nhãn mốc thời gian sang số ngày một lần cho cả hàng
body = raw.iloc[r_pid + 1:]
pid = pd.to_numeric(body.iloc[:, c_pid], errors="coerce").astype("Int64")
rows = pid.isin(master["PID"]).to_numpy()                # chỉ giữ cohort, loại 3 PID thừa (TBC, DTT, VBN)

def follow_long(name: str, metric: str) -> pd.DataFrame:
    target, picked = norm(metric), {}
    for c in raw.columns:
        if metric_row[c] == target and pd.notna(day_row[c]):
            picked.setdefault(int(day_row[c]), c)        # mỗi Day chỉ lấy cột đầu tiên gặp
    wide = body.loc[rows, list(picked.values())].copy()
    wide.columns = list(picked.keys())
    wide.insert(0, "PID", pid[rows].to_numpy())
    return to_long(wide, name)

follow_longs = [follow_long(n, m) for n, m in FOLLOW_FEATURES.items()]

# ============================== LUỒNG XỬ LÝ ==============================
# --- Chuẩn hoá dữ liệu và reshape ---
df = pd.concat(follow_longs + clin_longs + [il6_long], ignore_index=True)
df["Patient"] = df["PID"].map(pid2pat).astype(str)       # Patient là chuỗi BNxx
df["Diagnosis"] = df["PID"].map(pid2dx)
df["Day"] = df["Day"].astype(int)                        # Set Day là dạng int
df = df.reindex(columns=COLS)

# MRD/Blast chỉ áp dụng cho ALL, NHL là "không áp dụng" nên để NaN
df.loc[df["Diagnosis"].ne("ALL"), ALL_ONLY] = np.nan

# --- Nồng độ/số lượng tế bào không thể âm -> NaN ---
df[FEATURES] = df[FEATURES].astype(float).mask(lambda x: x < 0)

# --- Bỏ giá trị mang biến đều NaN (không có thông tin) ---
df = df.dropna(subset=FEATURES, how="all")

# --- Xử lý trùng (Patient, Day) bằng mean và sort theo thời gian ---
true_conflicts = {f: int(n) for f in FEATURES if (n := df.dropna(subset=[f]).duplicated(["Patient", "Day"]).sum())}
df = (df.groupby(["Patient", "Day", "Diagnosis"], as_index=False)[FEATURES].mean()
        .sort_values(["Patient", "Day"], ignore_index=True))

# --- Thêm đặc trưng hỗ trợ model ---
if ADD_DT:
    df["dt"] = df.groupby("Patient")["Day"].diff().fillna(0).astype(int)
if ADD_MASKS:
    df[[f"{c}_obs" for c in FEATURES]] = df[FEATURES].notna().astype("int8").to_numpy()

# --- KIỂM TRA TRƯỚC EXPORT ---
assert not df.duplicated(["Patient", "Day"]).any() and (df[FEATURES].min() >= 0).all()
assert df["Day"].dtype.kind == "i" and pd.api.types.is_string_dtype(df["Patient"])
assert df.groupby("Patient")["Day"].apply(lambda d: d.is_monotonic_increasing).all()

# ============================== TỔNG HỢP ==============================
print(f"Shape: {df.shape} | Patients: {df['Patient'].nunique()} | Số mốc/bệnh nhân: "
      f"min={df.groupby('Patient').size().min()}, median={int(df.groupby('Patient').size().median())}, max={df.groupby('Patient').size().max()}")
print("Trùng thật (cùng biến, cùng ngày) đã lấy mean:", true_conflicts or "không có")
is_all = df["Diagnosis"].eq("ALL")
rep = pd.DataFrame({"missing_%": (df[FEATURES].isna().mean() * 100).round(2),
                    "n_observed": df[FEATURES].notna().sum()})
rep["missing_%_in_ALL"] = np.where(rep.index.isin(ALL_ONLY), (df.loc[is_all, FEATURES].isna().mean() * 100).round(2), np.nan)
print("\nMissing theo từng cột \n" + rep.to_string())

df.to_csv(OUT, index=False)
df.head(12)

Shape: (181, 26) | Patients: 16 | Số mốc/bệnh nhân: min=10, median=12, max=12
Trùng thật (cùng biến, cùng ngày) đã lấy mean: {'IL6_pg_ml': 1}

Missing theo từng cột 
                    missing_%  n_observed  missing_%_in_ALL
B_cell_PB_cells_ml      12.71         158               NaN
B_cell_PB_pct           12.71         158               NaN
CAR_T_PB_cells_ul       21.55         142               NaN
CAR_T_PB_pct            21.55         142               NaN
B_cell_BM_cells_ml      83.43          30               NaN
B_cell_BM_pct           83.43          30               NaN
CAR_T_BM_cells_ul       83.43          30               NaN
CAR_T_BM_pct            83.43          30               NaN
MRD_pct                 88.40          21             76.14
Blast_pct               84.53          28             68.18
IL6_pg_ml               77.35          41               NaN


,Patient,Day,Diagnosis,B_cell_PB_cells_ml,B_cell_PB_pct,CAR_T_PB_cells_ul,CAR_T_PB_pct,B_cell_BM_cells_ml,B_cell_BM_pct,CAR_T_BM_cells_ul,...,B_cell_PB_pct_obs,CAR_T_PB_cells_ul_obs,CAR_T_PB_pct_obs,B_cell_BM_cells_ml_obs,B_cell_BM_pct_obs,CAR_T_BM_cells_ul_obs,CAR_T_BM_pct_obs,MRD_pct_obs,Blast_pct_obs,IL6_pg_ml_obs
0,BN01,0,ALL,3210.0,0.11,NaN,NaN,NaN,NaN,NaN,...,1,0,0,0,0,0,0,0,1,1
1,BN01,3,ALL,4260.0,0.19,0.649635,0.2415,NaN,NaN,NaN,...,1,1,1,0,0,0,0,0,0,0
2,BN01,6,ALL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,1
3,BN01,7,ALL,1210.0,0.03,143.104964,24.0100,NaN,NaN,NaN,...,1,1,1,0,0,0,0,0,0,0
4,BN01,8,ALL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176,BN18,21,NHL,301.0,0.02,357.936852,47.2836,NaN,NaN,NaN,...,1,1,1,0,0,0,0,0,0,0
177,BN18,30,NHL,0.0,0.00,67.269740,4.9102,NaN,NaN,NaN,...,1,1,1,0,0,0,0,0,0,0
178,BN18,60,NHL,100.0,0.00,5.570000,0.5400,NaN,NaN,NaN,...,1,1,1,0,0,0,0,0,0,0
179,BN18,90,NHL,651.0,0.04,3.140000,0.3400,601.0,0.02,4.14,...,1,1,1,1,1,1,1,0,0,0
